In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pytorch_lightning as pl
from sentence_transformers import SentenceTransformer
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from pytorch_lightning.callbacks import Callback
import pandas as pd
import pickle
from pytorch_lightning.callbacks import ModelCheckpoint

D:\Anaconda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.get_device_name(0)

'NVIDIA GeForce RTX 3060 Laptop GPU'

In [36]:
items = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/books_data_with_new_id.csv')
# movies = pd.read_csv('processed_dataset/MovieLens-1M/movies/movies_movielens.csv')

full_ratings = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/no dup new/amazon_books_ratings_full_filtered_fixed.csv')
train_ratings = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/no dup new/amazon_books_ratings_train_filtered_fixed.csv')
val_ratings = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/no dup new/amazon_books_ratings_val_filtered_fixed.csv')
test_ratings = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/no dup new/amazon_books_ratings_test_filtered_fixed.csv')

In [164]:
full_ratings

,Unnamed: 0.1,Unnamed: 0,item_id,Title,user_id,profileName,rating,timestamp
0,517468,2971446,B000G167FA,Silver Pennies,AZUNT3QP2CWTL,"Ellen C. Falkenberry ""ellenf""",5.0,-1
1,346461,2006209,B00005O4HA,Playing for the Ashes,A3RTKL9KB8KLID,Stan Vernooy,5.0,840240000
2,491168,2808945,B000NWR0W6,and ladies of the club,A3RTKL9KB8KLID,Stan Vernooy,5.0,854150400
3,371922,2144019,B000MVVYNO,Bethlehem Road,A3RTKL9KB8KLID,Stan Vernooy,4.0,862358400
4,381133,2196179,B0007E212E,"Hercules, My Shipmate",A3TEH90X39WC8F,"Stuart W. Mirsky ""swm""",5.0,863308800
...,...,...,...,...,...,...,...,...
330987,391414,2253077,1844560333,Pride and Prejudice,A2GDT5QQSFZD14,Joy Hilda Handley,5.0,1362268800
330988,365798,2110656,1593355548,Wuthering Heights,A2GDT5QQSFZD14,Joy Hilda Handley,5.0,1362268800
330989,365797,2110655,1593355548,Wuthering Heights,A2DKTZMMG3JHN4,K. Burns,4.0,1362268800
330990,205310,1188787,1593355548,Wuthering Heights,A2GDT5QQSFZD14,Joy Hilda Handley,5.0,1362268800


In [21]:
items

,Title,description,authors,image,previewLink,publisher,publishedDate,infoLink,categories,ratingsCount,item_id
0,Its Only Art If Its Well Hung!,NaN,['Julie Strain'],http://books.google.com/books/content?id=DykPA...,http://books.google.nl/books?id=DykPAAAACAAJ&d...,NaN,1996,http://books.google.nl/books?id=DykPAAAACAAJ&d...,['Comics & Graphic Novels'],NaN,1882931173
1,Dr. Seuss: American Icon,Philip Nel takes a fascinating look into the k...,['Philip Nel'],http://books.google.com/books/content?id=IjvHQ...,http://books.google.nl/books?id=IjvHQsCn_pgC&p...,A&C Black,1/1/2005,http://books.google.nl/books?id=IjvHQsCn_pgC&d...,['Biography & Autobiography'],NaN,826414346
2,Wonderful Worship in Smaller Churches,This resource includes twelve principles in un...,['David R. Ray'],http://books.google.com/books/content?id=2tsDA...,http://books.google.nl/books?id=2tsDAAAACAAJ&d...,NaN,2000,http://books.google.nl/books?id=2tsDAAAACAAJ&d...,['Religion'],NaN,829814000
3,Whispers of the Wicked Saints,Julia Thomas finds her life spinning out of co...,['Veronica Haddon'],http://books.google.com/books/content?id=aRSIg...,http://books.google.nl/books?id=aRSIgJlq6JwC&d...,iUniverse,2005-02,http://books.google.nl/books?id=aRSIgJlq6JwC&d...,['Fiction'],NaN,595344550
4,"Nation Dance: Religion, Identity and Cultural ...",NaN,['Edward Long'],NaN,http://books.google.nl/books?id=399SPgAACAAJ&d...,NaN,3/1/2003,http://books.google.nl/books?id=399SPgAACAAJ&d...,NaN,NaN,253338352
...,...,...,...,...,...,...,...,...,...,...,...
212399,The Orphan Of Ellis Island (Time Travel Advent...,"During a school trip to Ellis Island, Dominick...",['Elvira Woodruff'],http://books.google.com/books/content?id=J7M-N...,http://books.google.com/books?id=J7M-NwAACAAJ&...,Scholastic Paperbacks,6/1/2000,http://books.google.com/books?id=J7M-NwAACAAJ&...,['Juvenile Fiction'],2.0,590482467
212400,Red Boots for Christmas,Everyone in the village of Friedensdorf is hap...,NaN,http://books.google.com/books/content?id=3n8k6...,http://books.google.com/books?id=3n8k6wl4BbYC&...,NaN,1995,http://books.google.com/books?id=3n8k6wl4BbYC&...,['Juvenile Fiction'],NaN,570047870
212401,Mamaw,"Give your Mamaw a useful, beautiful and though...",['Wild Wild Cabbage'],NaN,http://books.google.com/books?id=zytVswEACAAJ&...,NaN,1/17/2018,http://books.google.com/books?id=zytVswEACAAJ&...,NaN,NaN,B000OVF7JY
212402,The Autograph Man,Alex-Li Tandem sells autographs. His business ...,['Zadie Smith'],http://books.google.com/books/content?id=JM6YV...,http://books.google.com/books?id=JM6YVPx_clMC&...,Vintage,8/12/2003,https://play.google.com/store/books/details?id...,['Fiction'],19.0,1402508735


In [37]:
items = items.dropna(subset=['item_id'])


In [38]:
duplicated_items = items[items.duplicated(subset='item_id', keep=False)]
duplicated_items

,Title,description,authors,image,previewLink,publisher,publishedDate,infoLink,categories,ratingsCount,item_id


In [84]:
def generate_user_texts_with_history(items, ratings):
    # Initialize user histories with an empty list for each unique user_id
    user_histories = {user_id: [] for user_id in ratings['user_id'].unique()}
    user_texts = []

    # Convert relevant columns to dictionaries for faster access
    items_dict = items.set_index('item_id')[['Title', 'categories', 'authors']].to_dict('index')

    for _, row in ratings.iterrows():
        user_id = row['user_id']
        item_id = row['item_id']
        profile_name = row['profileName']  # Get the profile name directly from the ratings DataFrame

        # Prepare the user's history (only the last 3 items)
        history_items = []
        for mid in user_histories[user_id][-3:]:
            if mid in items_dict and items_dict[mid]:
                title = items_dict[mid]['Title']
                category = items_dict[mid]['categories']
                author = items_dict[mid]['authors']

                # Clean and format the category
                if pd.notna(category):
                    category = category.strip("[]'\"")
                # if pd.notna(author):
                #     author = author.strip("[]'\"")
                # Build the history string based on the available data
                # history_entry = f"{title}"
                history_entry = f"{category}"

                # if pd.notna(author):
                #     history_entry += f"author: {author}"
                # if pd.notna(category):
                #     history_entry += f" [SEP] category: {category}"
                if history_entry != "":
                    history_items.append(history_entry)
        history_str = ", ".join(history_items)

        # Combine user history into the final text format
        if history_str:
            combined_features = f"[USER_PROFILE] profileName: {profile_name} [SEP] category: {history_str}"
        else:
            combined_features = f"[USER_PROFILE] profileName: {profile_name}"

        user_texts.append(combined_features)

        # Update the user history after generating combined features, only if the item_id is valid
        if item_id in items_dict and items_dict[item_id]:
            user_histories[user_id].append(item_id)

    return user_texts

# def generate_user_texts_with_history(items, ratings):
#     user_histories = {user_id: [] for user_id in ratings['user_id'].unique()}
#     user_texts = []
#
#     # Convert relevant columns to dictionaries for faster access
#     items_dict = items.set_index('item_id')[['Title', 'categories', 'authors']].to_dict('index')
#
#     for _, row in ratings.iterrows():
#         user_id = row['user_id']
#         item_id = row['item_id']
#
#         profile_name = row['profileName']  # Get the profile name directly from the ratings DataFrame
#
#         # Append the user's history (only the last 3 movies)
#         history_items = []
#         for mid in user_histories[user_id][-3:]:
#             if mid in items_dict and items_dict[mid]:
#                 title = items_dict[mid]['Title']
#                 category = items_dict[mid]['categories']
#                 author = items_dict[mid]['authors']
#                 if pd.notna(category):  # Only include category if it is not NaN
#                     category = category.strip("[]'\"")
#                     history_items.append(f"title: {title} [SEP] author: {author} [SEP] category: {category}")
#                 else:
#                     history_items.append(f"title: {title}")
#
#         history_str = ", ".join(history_items)
#
#         # Combine user history into the final text format
#         if history_str:
#             combined_features = f"profileName: {profile_name} [SEP] history: {history_str}"
#         else:
#             combined_features = f"profileName: {profile_name}"
#
#         user_texts.append(combined_features)
#
#         # Update the user history after generating combined features, only if the item_id is valid
#         if item_id in items_dict and items_dict[item_id]:
#             user_histories[user_id].append(item_id)
#
#     return user_texts
# def generate_user_texts_with_history(items, ratings):
#     user_histories = {user_id: [] for user_id in ratings['user_id'].unique()}
#     user_texts = []
#
#     # Convert relevant columns to dictionaries for faster access
#     items_dict = items.set_index('item_id')[['Title', 'categories']].to_dict('index')
#
#     for _, row in ratings.iterrows():
#         user_id = row['user_id']
#         item_id = row['item_id']
#
#         profile_name = row['profileName']  # Get the profile name directly from the ratings DataFrame
#
#         # Append the user's history (only the last 3 movies)
#         history_items = []
#         for mid in user_histories[user_id][-3:]:
#             if mid in items_dict and items_dict[mid]:
#                 title = items_dict[mid]['Title']
#                 category = items_dict[mid]['categories']
#                 if pd.notna(category):  # Only include category if it is not NaN
#                     category = category.strip("[]'\"")
#                     history_items.append(f"{category}")
#                 # else:
#                 #     history_items.append(f"title: {title}")
#
#         history_str = ", ".join(history_items)
#
#         # Combine user history into the final text format
#         if history_str:
#             combined_features = f"profileName: {profile_name} [SEP] category: {history_str}"
#         else:
#             combined_features = f"profileName: {profile_name}"
#
#         user_texts.append(combined_features)
#
#         # Update the user history after generating combined features, only if the item_id is valid
#         if item_id in items_dict and items_dict[item_id]:
#             user_histories[user_id].append(item_id)
#
#     return user_texts


In [86]:
train_user_texts = generate_user_texts_with_history(items, train_ratings)
val_user_texts = generate_user_texts_with_history(items, val_ratings)
test_user_texts = generate_user_texts_with_history(items, test_ratings)
test_user_texts[:100]

['[USER_PROFILE] profileName: Ellen C. Falkenberry "ellenf"',
 '[USER_PROFILE] profileName: Charles Slovenski',
 '[USER_PROFILE] profileName: N. Sausser "pucksau"',
 '[USER_PROFILE] profileName: Mary Whipple',
 '[USER_PROFILE] profileName: fawls@erols.com',
 '[USER_PROFILE] profileName: fawls@erols.com [SEP] category: Fiction',
 '[USER_PROFILE] profileName: fawls@erols.com [SEP] category: Fiction, Fiction',
 '[USER_PROFILE] profileName: Sai Li',
 '[USER_PROFILE] profileName: Sai Li [SEP] category: Fiction',
 '[USER_PROFILE] profileName: Sai Li [SEP] category: Fiction, Fiction',
 '[USER_PROFILE] profileName: P. Meltzer',
 '[USER_PROFILE] profileName: Michael Battaglia',
 '[USER_PROFILE] profileName: Michael Battaglia [SEP] category: Fiction',
 '[USER_PROFILE] profileName: Gerald Lipsky',
 '[USER_PROFILE] profileName: Steve Sailer',
 '[USER_PROFILE] profileName: nickoli@rmi.net',
 '[USER_PROFILE] profileName: nickoli@rmi.net [SEP] category: Fiction',
 '[USER_PROFILE] profileName: Mark Sh

In [170]:
len(train_user_texts)

262296

In [87]:
import pandas as pd

# def generate_last_user_texts_with_history(items, val_ratings):
#     user_histories = {}
#     last_user_texts = {}
#
#     # Convert items to a dictionary for faster access
#     items_dict = items.set_index('item_id')[['Title', 'categories']].to_dict('index')
#
#     # Process the val_ratings
#     for _, row in val_ratings.iterrows():
#         user_id = row['user_id']
#         item_id = row['item_id']
#         profile_name = row['profileName']  # Get the profile name directly from the ratings DataFrame
#
#         # Initialize user history if not already done
#         if user_id not in user_histories:
#             user_histories[user_id] = []
#
#         # Generate the user's history (only the last 3 items)
#         history_items = []
#         for mid in user_histories[user_id][-3:]:
#             if mid in items_dict:
#                 title = items_dict[mid].get('Title', '').strip()
#                 category = items_dict[mid]['categories']
#
#                 if pd.notna(category):  # Only include category if it is not NaN
#                     category = category.strip("[]'\"")
#                     history_items.append(f"title: {title} [SEP] category: {category}")
#                 else:
#                     history_items.append(f"title: {title}")
#
#         history_str = ", ".join(history_items)
#
#         # Combine history into the final text format
#         if history_str:
#             combined_features = f"profileName: {profile_name} [SEP] history: {history_str}"
#         else:
#             combined_features = f"profileName: {profile_name} [SEP] history: None"
#
#         # Update the dictionary to keep the last text for each user
#         last_user_texts[user_id] = combined_features
#
#         # Update the user history after generating combined features
#         if item_id in items_dict:  # Ensure item exists in the dictionary
#             user_histories[user_id].append(item_id)
#
#     return last_user_texts

# def generate_last_user_texts_with_history(items, val_ratings):
#     user_histories = {}
#     last_user_texts = {}
#
#     # Convert items to a dictionary for faster access
#     items_dict = items.set_index('item_id')[['Title', 'categories']].to_dict('index')
#
#     # Process the val_ratings
#     for _, row in val_ratings.iterrows():
#         user_id = row['user_id']
#         item_id = row['item_id']
#         profile_name = row['profileName']  # Get the profile name directly from the ratings DataFrame
#
#         # Initialize user history if not already done
#         if user_id not in user_histories:
#             user_histories[user_id] = []
#
#         # Generate the user's history (only the last 3 items)
#         history_items = []
#         for mid in user_histories[user_id][-3:]:
#             if mid in items_dict:
#                 title = items_dict[mid].get('Title', '').strip()
#                 category = items_dict[mid]['categories']
#
#                 if pd.notna(category):  # Only include category if it is not NaN
#                     category = category.strip("[]'\"")
#                     history_items.append(f"{category}")
#
#         history_str = ", ".join(history_items)
#
#         # Combine history into the final text format
#         if history_str:
#             combined_features = f"profileName: {profile_name} [SEP] category: {history_str}"
#         else:
#             combined_features = f"profileName: {profile_name}"
#
#         # Update the dictionary to keep the last text for each user
#         last_user_texts[user_id] = combined_features
#
#         # Update the user history after generating combined features
#         if item_id in items_dict:  # Ensure item exists in the dictionary
#             user_histories[user_id].append(item_id)
#
#     return last_user_texts


def generate_last_user_texts_with_history(items, val_ratings):
    user_histories = {}
    last_user_texts = {}

    # Convert items to a dictionary for faster access
    items_dict = items.set_index('item_id')[['Title', 'categories', 'authors']].to_dict('index')

    # Process the val_ratings
    for _, row in val_ratings.iterrows():
        user_id = row['user_id']
        item_id = row['item_id']
        profile_name = row['profileName']  # Get the profile name directly from the ratings DataFrame

        # Initialize user history if not already done
        if user_id not in user_histories:
            user_histories[user_id] = []

        # Generate the user's history (only the last 3 items)
        history_items = []
        for mid in user_histories[user_id][-3:]:
            if mid in items_dict:
                title = items_dict[mid].get('Title', '').strip()
                category = items_dict[mid]['categories']
                author = items_dict[mid]['authors']

                # Clean and format the category
                if pd.notna(category):
                    category = category.strip("[]'\"")
                if pd.notna(author):
                    author = author.strip("[]'\"")
                # Build the history string based on the available data
                # if category != "":
                history_entry = f"{category}"
                # if pd.notna(author):
                #     history_entry += f" [SEP] author: {author}"
                # if pd.notna(category):
                #     if history_entry:
                #         history_entry += f" [SEP] category: {category}"
                #     else:
                #         history_entry += f"category: {category}"

                if history_entry:
                    history_items.append(history_entry)

        history_str = ", ".join(history_items)

        # Combine history into the final text format
        if history_str:
            combined_features = f"profileName: {profile_name} [SEP] category: {history_str}"
        else:
            combined_features = f"profileName: {profile_name}"

        # Update the dictionary to keep the last text for each user
        last_user_texts[user_id] = combined_features

        # Update the user history after generating combined features
        if item_id in items_dict:  # Ensure item exists in the dictionary
            user_histories[user_id].append(item_id)

    return last_user_texts

# Generate the last user texts for the validation data
val_last_user_texts = generate_last_user_texts_with_history(items, val_ratings)

In [51]:
len(test_user_texts)

36027

In [88]:
val_last_user_texts

{'A3RTKL9KB8KLID': 'profileName: Stan Vernooy [SEP] category: Fiction, Fiction',
 'A3TEH90X39WC8F': 'profileName: Stuart W. Mirsky "swm" [SEP] category: Fiction',
 'A3SOB0CMUBK6XJ': 'profileName: fawls@erols.com',
 'A2QBHNK9H2SVRJ': 'profileName: Angela Linton "Angie" [SEP] category: Fiction',
 'A2YUZKPLUYQDKV': 'profileName: Michael Battaglia [SEP] category: Fiction, Fiction, Fiction',
 'A3927BH5H75LII': 'profileName: Sai Li [SEP] category: Manhattan (New York, N.Y.), Fiction, Philosophy',
 'A2FR8GG77M4TP7': 'profileName: P. Meltzer [SEP] category: Computers',
 'A1GBOCJ943SP8R': 'profileName: Steve Sailer [SEP] category: Fiction',
 'AXOA9OI962P0Q': 'profileName: nickoli@rmi.net [SEP] category: Fiction',
 'ARG7WTYP8L66M': 'profileName: James Paris "Tarnmoor" [SEP] category: Blandings Castle (England : Imaginary place), Catalogs, Union',
 'A2T28ETOO7OBE': 'profileName: Travis Cottreau [SEP] category: Electronic books, Fiction, Ahab, Captain (Fictitious character)',
 'A1UMFRK5YTITOY': 'p

In [207]:
print(val_last_user_texts.get('A1BUCXZTW926PL'))

None


In [149]:
print(test_user_texts[2])

profileName: N. Sausser "pucksau"


In [15]:
# # Save user embeddings locally
# with open('train_user_texts.pkl', 'wb') as f:
#     pickle.dump(train_user_texts, f)
#
# print("Train user embeddings saved successfully.")
#
# with open('val_user_texts.pkl', 'wb') as f:
#     pickle.dump(val_user_texts, f)
#
# print("Validation user embeddings saved successfully.")
#
# with open('test_user_texts.pkl', 'wb') as f:
#     pickle.dump(test_user_texts, f)
#
# print("Test user embeddings saved successfully.")

In [16]:
# # Load user texts from file
# with open('./text_for_embeddings/last_three_history/train_user_texts.pkl', 'rb') as f:
#     train_user_texts = pickle.load(f)
# print("Train user text loaded successfully.")
#
# with open('./text_for_embeddings/last_three_history/val_user_texts.pkl', 'rb') as f:
#     val_user_texts = pickle.load(f)
# print("Validation user text loaded successfully.")
#
# with open('./text_for_embeddings/last_three_history/test_user_texts.pkl', 'rb') as f:
#     test_user_texts = pickle.load(f)
# print("Test user text loaded successfully.")

In [90]:
# Combine movie features into a single string for each movie
# movies['movie_features'] = 'title: ' + movies['title'] + ' [SEP] genres: ' + movies['genres']
items['categories'].fillna('', inplace=True)
items['authors'].fillna('', inplace=True)
items['cleaned_categories'] = items['categories'].str.strip('[]').str.replace("'", '')
# items['cleaned_authors'] = items['authors'].str.strip('[]').str.replace("'", '')


# Define book_features with the specific condition
items['book_features'] = items.apply(
    lambda row: (
        f"[BOOK_DETAIL] title: {row['Title']} [SEP] category: {row['cleaned_categories']}"
        if row['cleaned_categories'] else
        f"[BOOK_DETAIL] title: {row['Title']}"
    ),
    axis=1
)

# items['book_features'] = items.apply(
#     lambda row: f"category: {row['cleaned_categories']}" if row['cleaned_categories'] else f"title: {row['Title']}",
#     axis=1
# )
# items['book_features'] = items.apply(
#     lambda row: f"title: {row['Title']} [SEP] author: {row['cleaned_authors']} [SEP] category: {row['cleaned_categories']}"
#     if row['cleaned_categories'] and row['cleaned_authors']
#     else (
#         f"title: {row['Title']} [SEP] author: {row['cleaned_authors']}" if row['cleaned_authors']
#         else (
#             f"title: {row['Title']} [SEP] category: {row['cleaned_categories']}" if row['cleaned_categories']
#             else f"title: {row['Title']}"
#         )
#     ),
#     axis=1
# )

# movies['movie_features'] = '[MOVIE_DETAIL] genres: ' + movies['genres']
items['book_features']

C:\Users\Hooman\AppData\Local\Temp\ipykernel_6592\1570470146.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  items['categories'].fillna('', inplace=True)
C:\Users\Hooman\AppData\Local\Temp\ipykernel_6592\1570470146.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example

0         [BOOK_DETAIL] title: Its Only Art If Its Well ...
1         [BOOK_DETAIL] title: Dr. Seuss: American Icon ...
2         [BOOK_DETAIL] title: Wonderful Worship in Smal...
3         [BOOK_DETAIL] title: Whispers of the Wicked Sa...
4         [BOOK_DETAIL] title: Nation Dance: Religion, I...
                                ...                        
212399    [BOOK_DETAIL] title: The Orphan Of Ellis Islan...
212400    [BOOK_DETAIL] title: Red Boots for Christmas [...
212401                           [BOOK_DETAIL] title: Mamaw
212402    [BOOK_DETAIL] title: The Autograph Man [SEP] c...
212403    [BOOK_DETAIL] title: Student's Solutions Manua...
Name: book_features, Length: 206705, dtype: object

In [242]:
items['cleaned_categories']

0           Comics & Graphic Novels
1         Biography & Autobiography
2                          Religion
3                           Fiction
4                                  
                    ...            
212399             Juvenile Fiction
212400             Juvenile Fiction
212401                             
212402                      Fiction
212403                             
Name: cleaned_categories, Length: 206705, dtype: object

In [12]:
items['book_features']

0         title: Its Only Art If Its Well Hung! [SEP] au...
1         title: Dr. Seuss: American Icon [SEP] author: ...
2         title: Wonderful Worship in Smaller Churches [...
3         title: Whispers of the Wicked Saints [SEP] aut...
4         title: Nation Dance: Religion, Identity and Cu...
                                ...                        
212399    title: The Orphan Of Ellis Island (Time Travel...
212400    title: Red Boots for Christmas [SEP] category:...
212401         title: Mamaw [SEP] author: Wild Wild Cabbage
212402    title: The Autograph Man [SEP] author: Zadie S...
212403    title: Student's Solutions Manual for Johnson/...
Name: book_features, Length: 206705, dtype: object

In [91]:
# Create a dictionary for fast lookup
item_features_dict = items.set_index('item_id')['book_features'].to_dict()

# Create lists of user and item texts
item_texts = [item_features_dict[itemId] for itemId in full_ratings['item_id'].unique()]

# Create a mapping from userId and movieId to indices
item_id_to_idx = {itemId: idx for idx, itemId in enumerate(full_ratings['item_id'].unique())}

# Map userId and movieId in ratings_df to indices
train_ratings['item_idx'] = train_ratings['item_id'].map(item_id_to_idx)

# Map userId and movieId in ratings_val to indices
val_ratings['item_idx'] = val_ratings['item_id'].map(item_id_to_idx)

# Map userId and movieId in ratings_test to indices
test_ratings['item_idx'] = test_ratings['item_id'].map(item_id_to_idx)

# Extract user indices, item indices, and ratings
train_item_indices = torch.LongTensor(train_ratings['item_idx'].values).to(device)
train_labels = torch.FloatTensor(train_ratings['rating'].values).to(device)

# Extract user indices, item indices, and ratings for validation
val_item_indices = torch.LongTensor(val_ratings['item_idx'].values).to(device)
val_labels = torch.FloatTensor(val_ratings['rating'].values).to(device)

# Extract user indices, item indices, and ratings for test
test_item_indices = torch.LongTensor(test_ratings['item_idx'].values).to(device)
test_labels = torch.FloatTensor(test_ratings['rating'].values).to(device)


In [79]:
len(test_labels)

490

In [80]:
test_ratings

,rating,title,text,images,asin,parent_asin,user_id,timestamp,verified_purchase,helpful_vote,parent_asin_idx
0,4,Good hold with slight shine,This product is tacky so it has a good hold if...,[{'small_image_url': 'https://images-na.ssl-im...,B07K1QJH5P,B07K1QJH5P,AE3KLVXGZPANXE5XLXYKHTVAZ3FQ,1.610920e+12,False,0,1846
1,4,Pretty decent product but do your research,"Overall, this product isn’t horrible compared ...",[{'small_image_url': 'https://images-na.ssl-im...,B01N7A5AGF,B01N7A5AGF,AE3KLVXGZPANXE5XLXYKHTVAZ3FQ,1.613920e+12,False,2,722
2,5,Take me back to the 90s!,I had a set of headbands JUST like these in th...,[{'small_image_url': 'https://images-na.ssl-im...,B08QHP717Z,B08QHP717Z,AE3KLVXGZPANXE5XLXYKHTVAZ3FQ,1.614550e+12,False,0,230
3,4,"Nice Cleanser That Leaves No Oily Residue, But...",This Neutrogena cleansing oil is nice and does...,[],B00U2VQZC4,B00U2VQZC4,AE3PLZHW6NXWBMZ76TDVFQG2MJFA,1.441240e+12,False,0,3340
4,5,Eye Replacement Head Fits Glo Pro Perfectly & ...,I’m using the Glo Pro eye attachment head with...,[{'small_image_url': 'https://images-na.ssl-im...,B06XD3SXQ8,B09J5TZ7HL,AE3PLZHW6NXWBMZ76TDVFQG2MJFA,1.539880e+12,False,16,1589
...,...,...,...,...,...,...,...,...,...,...,...
485,5,Love the convenient handle,This does a great job exfoliating the skin and...,[{'small_image_url': 'https://images-na.ssl-im...,B08B5FJMHM,B08B5FJMHM,AHY2TURQPNIDXZGH2CMQLZ343YMQ,1.598400e+12,False,0,10905
486,4,I can use this even with my sensitive eyes and...,I have sensitive skin and eyes. Without the r...,[],B01CYTUXHO,B01CYTUXHO,AHYOSWORVZFXM5QMRIAW3JTTFFIQ,1.477420e+12,False,1,3385
487,5,Big and thick with little sponge hearts!!!,These are great!! They are big and thick with ...,[],B01N2XUHTW,B01N2XUHTW,AHYOSWORVZFXM5QMRIAW3JTTFFIQ,1.493010e+12,False,0,1822
488,5,Sturdy!,Great tweezers!!<br />Sturdy set!!,[],B01GL4HV64,B01GL4HV64,AHYOSWORVZFXM5QMRIAW3JTTFFIQ,1.497500e+12,False,2,689


In [92]:
from torch.utils.data import Dataset, DataLoader

class CustomTextDataset(Dataset):
    def __init__(self, users, item_ids, ratings):
        self.users = users
        self.item_ids = item_ids
        self.ratings = ratings

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        users = self.users[idx]
        item_id = self.item_ids[idx]
        rating = self.ratings[idx]
        return users, item_id, rating

In [93]:
# Create DataLoader for training data
train_dataset = CustomTextDataset(train_user_texts, train_item_indices, train_labels)
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)

# Create DataLoader for validation data
val_dataset = CustomTextDataset(val_user_texts, val_item_indices, val_labels)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=True, drop_last=True)

# Create DataLoader for test data
test_dataset = CustomTextDataset(test_user_texts, test_item_indices, test_labels)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=True, drop_last=True)

In [94]:
class TwoTowerModel(pl.LightningModule):
    def __init__(self, user_model_name, item_model_name, embedding_size=384):
        super(TwoTowerModel, self).__init__()
        self.user_model = SentenceTransformer(user_model_name)
        self.item_model = SentenceTransformer(item_model_name)

        self.user_fc = nn.Linear(embedding_size, embedding_size)
        self.item_fc = nn.Linear(embedding_size, embedding_size)

        self.criterion = nn.MSELoss()
        self.epoch_losses = {'train_loss': [], 'val_loss': []}

    def forward(self, user_text, item_text):
        user_embedding = self.user_model.encode(user_text, convert_to_tensor=True).to(device)
        item_embedding = self.item_model.encode(item_text, convert_to_tensor=True).to(device)

        user_output = self.user_fc(user_embedding)
        item_output = self.item_fc(item_embedding)

        dot_product = torch.matmul(user_output.squeeze(), item_output.T)
        dot_product = 4 * torch.sigmoid(dot_product) + 1

        return dot_product

    def training_step(self, batch, batch_idx):
        users, items, ratings = batch

        items = [item_texts[i] for i in items.tolist()]

        preds = self(users, items)

        loss = self.criterion(preds, ratings)
        self.log('train_loss', loss)
        return loss

    def validation_step(self, batch, batch_idx):
        users, items, ratings = batch

        items = [item_texts[i] for i in items.tolist()]

        preds = self(users, items)

        loss = self.criterion(preds, ratings)
        self.log('val_loss', loss)
        return loss

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=1e-5)

class PrintLossesCallback(Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        train_loss = trainer.callback_metrics.get('train_loss')
        if train_loss is not None:
            pl_module.epoch_losses['train_loss'].append(train_loss.item())
            print(f"Epoch {trainer.current_epoch + 1}: Train Loss: {train_loss.item()}")

    def on_validation_epoch_end(self, trainer, pl_module):
        val_loss = trainer.callback_metrics.get('val_loss')
        if val_loss is not None:
            pl_module.epoch_losses['val_loss'].append(val_loss.item())
            print(f"Epoch {trainer.current_epoch + 1}: Val Loss: {val_loss.item()}")

In [95]:
# model = TwoTowerModel(user_model_name='paraphrase-MiniLM-L6-v2', item_model_name='paraphrase-MiniLM-L6-v2')
model = TwoTowerModel(user_model_name='paraphrase-MiniLM-L12-v2', item_model_name='paraphrase-MiniLM-L12-v2')

# Define the ModelCheckpoint callback
checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',  # Metric to monitor
    dirpath='checkpoints/',  # Directory to save the checkpoints
    filename='with-history-best-checkpoint',  # Filename for the best model
    save_top_k=1,  # Save only the top 1 model
    mode='min'  # Mode to save the best model (min for validation loss)
)

trainer = pl.Trainer(max_epochs=5, log_every_n_steps=1, callbacks=[PrintLossesCallback()], enable_progress_bar=True)
trainer.fit(model, train_dataloader, val_dataloader)

# Print losses after training completes
print("Epoch losses:")
for epoch in range(trainer.max_epochs):
    train_loss = model.epoch_losses['train_loss'][epoch] if epoch < len(model.epoch_losses['train_loss']) else 'N/A'
    val_loss = model.epoch_losses['val_loss'][epoch] if epoch < len(model.epoch_losses['val_loss']) else 'N/A'
    print(f"Epoch {epoch + 1}: Train Loss: {train_loss}, Val Loss: {val_loss}")

D:\Anaconda\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name       | Type                | Params | Mode 
-----------------------------------------------------------
0 | user_model | SentenceTransformer | 33.4 M | train
1 | item_model | SentenceTransformer | 33.4 M | train
2 | user_fc    | Linear              | 147 K  | train
3 | item_fc    | Linear              | 147 K  | train
4 | criterion  | MSELoss             | 0      | train
-----------------------------------------------------------
67.0 M    Trainable params
0         Non-trainable params
67.0 M    Total params
268.063   Total estimated model params siz

Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

D:\Anaconda\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:475: Your `val_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.
D:\Anaconda\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Sanity Checking DataLoader 0: 100%|██████████| 2/2 [00:00<00:00,  5.69it/s]Epoch 1: Val Loss: 2.426271438598633
                                                                           

D:\Anaconda\lib\site-packages\torch\nn\modules\loss.py:535: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 64])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
D:\Anaconda\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Epoch 0: 100%|██████████| 4098/4098 [07:21<00:00,  9.29it/s, v_num=3]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 510/510 [01:33<00:00,  5.43it/s]Epoch 1: Val Loss: 0.9798647165298462

Epoch 1: 100%|██████████| 4098/4098 [13:54<00:00,  4.91it/s, v_num=3]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 510/510 [01:25<00:00,  5.95it/s]Epoch 2: Val Loss: 0.9753478765487671

Epoch 2: 100%|██████████| 4098/4098 [14:38<00:00,  4.67it/s, v_num=3]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 510/510 [01:58<00:00,  4.30it/s]Epoch 3: Val Loss: 0.9739379286766052

Epoch 3: 100%|██████████| 4098/4098 [14:27<00:00,  4.72it/s, v_num=3]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 510/510 [01:58<00:00,  4.31it/s]Epoch 4: Val Loss: 0.9708672761917114

Epoch 4: 100%|██████████| 4098/4098 [14:01<00:00,  4.87it/s, v_num=3]
Validation: | 

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 4098/4098 [15:44<00:00,  4.34it/s, v_num=3]
Epoch losses:
Epoch 1: Train Loss: 0.9110240936279297, Val Loss: 2.426271438598633
Epoch 2: Train Loss: 0.598984956741333, Val Loss: 0.9798647165298462
Epoch 3: Train Loss: 0.8507875800132751, Val Loss: 0.9753478765487671
Epoch 4: Train Loss: 1.028587818145752, Val Loss: 0.9739379286766052
Epoch 5: Train Loss: 0.7506886720657349, Val Loss: 0.9708672761917114


In [20]:
model.epoch_losses

{'train_loss': [0.9391066431999207,
  1.0028347969055176,
  0.9369568824768066,
  0.9828064441680908,
  0.9409568905830383],
 'val_loss': [10.464865684509277,
  1.1707780361175537,
  1.138741135597229,
  1.120898962020874,
  1.1324385404586792,
  1.116618275642395]}

# Evaluation

In [96]:
# Assuming the training part has been done already, load the best model checkpoint


# best_model_path = './lightning_logs/paraphrase-MiniLM-L12-v2/not-binarized/user_title + store & item_ title + store  _ 5 epoch/checkpoints/epoch=4-step=435.ckpt'
best_model_path = './lightning_logs/version_3/checkpoints/epoch=4-step=20490.ckpt'

# best_model = TwoTowerModel.load_from_checkpoint(best_model_path, user_model_name='paraphrase-MiniLM-L6-v2', item_model_name='paraphrase-MiniLM-L6-v2').to(device)
best_model = TwoTowerModel.load_from_checkpoint(best_model_path, user_model_name='paraphrase-MiniLM-L12-v2', item_model_name='paraphrase-MiniLM-L12-v2').to(device)


D:\Anaconda\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## Calculations

In [97]:
def get_top_n_items_without_history_unseen_items(model, userId, n):
    # Ensure the model is in evaluation mode
    model.eval()

    # Get the user text for the given userId
    user_text = val_last_user_texts[userId]
    # print(user_text)
    # Encode the user text
    user_embedding = model.user_model.encode(user_text, convert_to_tensor=True).to(device)

    # Compute the scores (dot product between user embedding and each item embedding)
    user_output = model.user_fc(user_embedding).to(device)
    item_output = model.item_fc(full_items_embeddings).to(device)
    dot_product = torch.matmul(user_output, item_output.t()).squeeze()

    # Get items the user has seen in the training and validation data
    seen_items_train = train_ratings[train_ratings['user_id'] == userId]['item_id'].values
    seen_items_val = val_ratings[val_ratings['user_id'] == userId]['item_id'].values
    seen_items = set(np.concatenate((seen_items_train, seen_items_val)))
    # print(dot_product)
    # print(len(dot_product), len(seen_items))
    # Get the top n + len(seen_items) item indices and their scores
    # top_n_scores, top_n_indices = torch.topk(dot_product, n + len(seen_items))
    top_n_scores, top_n_indices = torch.topk(dot_product, n)

    # Map indices back to item IDs
    top_n_item_ids = [list(item_id_to_idx.keys())[list(item_id_to_idx.values()).index(idx.item())] for idx in top_n_indices]
    # print(top_n_item_ids)
    # Filter out seen items
    # unseen_top_n_item_ids = [item for item in top_n_item_ids if item not in seen_items]
    # print(unseen_top_n_item_ids[:n])
    # return unseen_top_n_item_ids[:n]
    # print(top_n_item_ids[:n])
    return top_n_item_ids[:n]


In [184]:
item_texts[:20]

['title: Silver Pennies',
 'title: Playing for the Ashes [SEP] category: Fiction',
 'title: and ladies of the club [SEP] category: Fiction',
 'title: Bethlehem Road [SEP] category: Fiction',
 'title: Hercules, My Shipmate [SEP] category: Fiction',
 'title: HERCULES, MY SHIPMATE [SEP] category: Fiction',
 'title: Guns, Germs, and Steel: The Fates of Human Societies [SEP] category: History',
 'title: The Pathfinder [SEP] category: Fiction',
 'title: The Pathfinder - The Works of J. Fenimore Cooper [SEP] category: United States',
 'title: The Pathfinder, [SEP] category: Fiction',
 "title: The pathfinder (The modern readers' series) [SEP] category: Bumppo, Natty (Fictitious character)",
 'title: Cold Sassy Tree',
 'title: The King Must Die (Cardinal Giant GC-78)',
 'title: Dance to the Piper [SEP] category: Biography & Autobiography',
 'title: Dance to the piper (A Bantam giant)',
 "title: NATHAN'S RUN. [SEP] category: Fiction",
 'title: Eight Months on Ghazzah Street [SEP] category: Ficti

In [98]:
# Assuming full_items_embeddings is already defined
full_items_embeddings = torch.stack([best_model.item_model.encode(item_text, convert_to_tensor=True) for item_text in item_texts[27]]).to(device)

In [21]:
item_texts[8]

'title: The Pathfinder - The Works of J. Fenimore Cooper [SEP] author: James Fenimore Cooper [SEP] category: United States'

## Type 0

In [99]:
def dcg(scores, k):
    scores = np.asfarray(scores)[:k]
    return np.sum(scores / np.log2(np.arange(2, scores.size + 2)))

def ndcg_at_k(labels, k):
    ideal_labels = sorted(labels, reverse=True)
    return dcg(labels, k) / dcg(ideal_labels, k)

def recall_at_k(labels, relevant_count, k):
    return np.sum(labels[:k]) / relevant_count

def mrr_at_k(labels, k):
    for i, label in enumerate(labels[:k]):
        if label == 1:
            return 1 / (i + 1)
    return 0

def evaluate_user_cf_model(model, test_data, k):
    ndcg_scores = []
    recall_scores = []
    mrr_scores = []

    # Get unique users
    unique_users = test_data['user_id'].unique()

    for user in unique_users:
        # print(user)
        # Get the top N items for the user, filtering out seen items
        recommended_items = get_top_n_items_without_history_unseen_items(model, user, k)
        # recommended_titles = [item_titles.get(item, "Unknown Title") for item in recommended_items]

        # print("Recommended items and their titles:")

        # for item, title in zip(recommended_items, recommended_titles):
        #     print(f"{item}: {title}")
        # recommended_items = ['1844560333', '1593355548', 'B000CPSU9G', '1901768600', 'B000NDSX6C', '158726398X', 'B000P3LVZA', 'B000N6DDJQ', 'B000L5XWTA', 'B0000CO4JZ']

        user_test_data = test_data[test_data['user_id'] == user]
        test_items = user_test_data['item_id'].values

        y_score = [1 if item in test_items else 0 for item in recommended_items]
        # print(y_score)
        ndcg = ndcg_at_k(y_score, k)
        recall = recall_at_k(y_score, len(test_items), k)
        mrr = mrr_at_k(y_score, k)

        ndcg_scores.append(ndcg)
        recall_scores.append(recall)
        mrr_scores.append(mrr)

    # avg_ndcg = np.mean(np.nan_to_num(ndcg_scores, nan=0.0))

    avg_ndcg = np.nanmean(ndcg_scores)
    avg_recall = np.nanmean(recall_scores)
    avg_mrr = np.nanmean(mrr_scores)

    return {
        'NDCG@{}'.format(k): avg_ndcg,
        'Recall@{}'.format(k): avg_recall,
        'MRR@{}'.format(k): avg_mrr,
    }

all_items = items['item_id'].unique()

eval_result = evaluate_user_cf_model(best_model, test_ratings, k=5)
print(eval_result)
eval_result = evaluate_user_cf_model(best_model, test_ratings, k=10)
print(eval_result)

C:\Users\Hooman\AppData\Local\Temp\ipykernel_6592\2329117254.py:7: RuntimeWarning: invalid value encountered in scalar divide
  return dcg(labels, k) / dcg(ideal_labels, k)


{'NDCG@5': 0.6899557471018413, 'Recall@5': 0.002116333315667172, 'MRR@5': 0.005650522317188983}
{'NDCG@10': 0.4379996905984457, 'Recall@10': 0.005554211342268338, 'MRR@10': 0.006138357527246416}


In [ ]:
{'NDCG@5': 0.749980415767912, 'Recall@5': 0.000774772528200495, 'MRR@5': 0.0028648306426084205}

In [235]:
# Find the top 5 most repeated item_id
top_5_item_ids = full_ratings['item_id'].value_counts().head(10)

top_5_item_ids


item_id
1844560333    4079
1593355548    3105
B000CPSU9G    2634
1901768600    1908
B000NDSX6C    1879
158726398X    1843
B000P3LVZA    1644
B000N6DDJQ    1408
B000L5XWTA    1123
B0000CO4JZ    1120
Name: count, dtype: int64

## Type 2

In [100]:
def evaluate_user_cf_model(model, test_data, train_data, val_data, all_items, k):
    ndcg_scores = []

    # Get unique users
    unique_users = test_data['user_id'].unique()

    for user in unique_users:
        # Get the top N items for the user, filtering out seen items
        recommended_items = get_top_n_items_without_history_unseen_items(model, user, k)
        # recommended_items = ['1844560333', '1593355548', 'B000CPSU9G', '1901768600', 'B000NDSX6C']

        # recommended_items = ['1844560333', '1593355548', 'B000CPSU9G', '1901768600', 'B000NDSX6C', '158726398X', 'B000P3LVZA', 'B000N6DDJQ', 'B000L5XWTA', 'B0000CO4JZ']

        user_test_data = test_data[test_data['user_id'] == user]
        test_items = user_test_data['item_id'].values

        y_score = [
            user_test_data[user_test_data['item_id'] == item]['rating'].values[0] if item in test_items else 2.5
            for item in recommended_items
        ]

        ndcg = ndcg_at_k(y_score, k)
        ndcg_scores.append(ndcg)

    avg_ndcg = np.nanmean(ndcg_scores)

    return {
        'NDCG@{}'.format(k): avg_ndcg
    }

all_items = items['item_id'].unique()
# Evaluate the model
eval_result = evaluate_user_cf_model(best_model, test_ratings, train_ratings, val_ratings, all_items, k=5)
print(eval_result)
eval_result = evaluate_user_cf_model(best_model, test_ratings, train_ratings, val_ratings, all_items, k=10)
print(eval_result)

{'NDCG@5': 0.9993542327471552}
{'NDCG@10': 0.998347837129736}


## Type 3

In [101]:
def evaluate_user_cf_model(model, test_data, k):
    ndcg_scores = []

    # Get unique users
    unique_users = test_data['user_id'].unique()

    for user in unique_users:
        # Get the top N items for the user, filtering out seen items
        recommended_items = get_top_n_items_without_history_unseen_items(model, user, k)
        # recommended_items = ['1844560333', '1593355548', 'B000CPSU9G', '1901768600', 'B000NDSX6C']
        # recommended_items = ['1844560333', '1593355548', 'B000CPSU9G', '1901768600', 'B000NDSX6C', '158726398X', 'B000P3LVZA', 'B000N6DDJQ', 'B000L5XWTA', 'B0000CO4JZ']

        user_test_data = test_data[test_data['user_id'] == user]
        test_items = user_test_data['item_id'].values
        # print(user)

        y_score = [
            user_test_data[user_test_data['item_id'] == item]['rating'].values[0] if item in test_items else 0
            for item in recommended_items
        ]

        ndcg = ndcg_at_k(y_score, k)
        ndcg_scores.append(ndcg)

    avg_ndcg = np.nanmean(ndcg_scores)

    return {
        'NDCG@{}'.format(k): avg_ndcg
    }

all_items = items['item_id'].unique()
# Evaluate the model
eval_result = evaluate_user_cf_model(best_model, test_ratings, k=5)
print(eval_result)
eval_result = evaluate_user_cf_model(best_model, test_ratings, k=10)
print(eval_result)

C:\Users\Hooman\AppData\Local\Temp\ipykernel_6592\2329117254.py:7: RuntimeWarning: invalid value encountered in scalar divide
  return dcg(labels, k) / dcg(ideal_labels, k)


{'NDCG@5': 0.6899557471018413}
{'NDCG@10': 0.4379996905984457}


## Type 1

In [ ]:
def evaluate_user_cf_model(model, test_data, train_data, val_data, all_items, k):
    ndcg_scores = []

    # Get unique users
    unique_users = test_data['user_id'].unique()

    for user in unique_users:
        # Get the top N items for the user, filtering out seen items
        recommended_items = get_top_n_items_without_history_unseen_items(model, user, k)

        user_test_data = test_data[test_data['user_id'] == user]
        test_items = user_test_data['item_id'].values
        print(user)
        # y_score = [
        #     user_test_data[user_test_data['item_id'] == item]['rating'].values[0] if item in test_items else 0
        #     for item in recommended_items
        # ]
        y_score = [
            1 if (item in test_items and user_test_data[user_test_data['item_id'] == item]['label'].values[0] == 1) else 0
            for item in recommended_items
        ]
        # y_score = [
        #     1 if (item in test_items and user_test_data[user_test_data['item'] == item]['label'].values[0] == 1) else 0
        #     for item in recommended_items
        # ]

        ndcg = ndcg_at_k(y_score, k)
        ndcg_scores.append(ndcg)

    # avg_ndcg = np.mean(np.nan_to_num(ndcg_scores, nan=0.0))
    avg_ndcg = np.nanmean(ndcg_scores)

    return {
        'NDCG@{}'.format(k): avg_ndcg
    }

all_items = movies['item_id'].unique()
# Evaluate the model
eval_result = evaluate_user_cf_model(best_model, test_ratings, train_ratings, val_ratings, all_items, k=5)
print(eval_result)
eval_result = evaluate_user_cf_model(best_model, test_ratings, train_ratings, val_ratings, all_items, k=5)
print(eval_result)